# LangChain — Chains, LCEL, and Retrievers

Master LangChain Expression Language for composable, production-grade LLM pipelines with custom chains, retrievers, and output parsers.

**Topics:** LCEL pipe operator, custom retrievers, output parsers, document transformers, complex chains

## 1. LCEL Basics — The Pipe Operator and Composability

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
import os

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.7, api_key=os.getenv('OPENAI_API_KEY', 'sk-...'))

# LCEL: chain components with | operator
simple_chain = (
    ChatPromptTemplate.from_template('Summarize this in one sentence: {text}')
    | llm
    | StrOutputParser()
)

# Multi-step chain: summarize → classify
summarize_prompt = ChatPromptTemplate.from_template('Summarize in one sentence: {text}')
classify_prompt = ChatPromptTemplate.from_template(
    'Classify this text as TECHNICAL or NON-TECHNICAL. Text: {summary}\nLabel:'
)

pipeline = (
    {'summary': summarize_prompt | llm | StrOutputParser(), 'text': RunnablePassthrough()}
    | classify_prompt
    | llm
    | StrOutputParser()
)

# Parallel execution with RunnableParallel
from langchain_core.runnables import RunnableParallel

parallel_chain = RunnableParallel({
    'summary': summarize_prompt | llm | StrOutputParser(),
    'keywords': ChatPromptTemplate.from_template('List 5 keywords from: {text}') | llm | StrOutputParser(),
    'sentiment': ChatPromptTemplate.from_template('Sentiment (POS/NEG/NEU) of: {text}') | llm | StrOutputParser(),
})

print('LCEL chain types:')
print('  simple_chain:   prompt | llm | parser')
print('  pipeline:       multi-step with intermediate values')
print('  parallel_chain: run multiple chains concurrently')
print()
print('LCEL benefits over legacy Chains:')
for benefit in ['Streaming support', 'Async by default', 'Batch processing', 'Easy inspection/debugging', 'Unified interface']:
    print(f'  ✓ {benefit}')

## 2. Custom Retriever Implementation

In [ ]:
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from typing import List
import httpx

class APIRetriever(BaseRetriever):
    """Retriever that fetches documents from an external API."""
    
    api_url: str
    top_k: int = 5
    
    def _get_relevant_documents(
        self,
        query: str,
        *,
        run_manager: CallbackManagerForRetrieverRun,
    ) -> List[Document]:
        response = httpx.get(self.api_url, params={'q': query, 'k': self.top_k})
        results = response.json()['results']
        return [Document(page_content=r['text'], metadata={'source': r['url']}) for r in results]

class WeightedEnsembleRetriever(BaseRetriever):
    """Combine multiple retrievers with custom weighting."""
    
    retrievers: list
    weights: list[float]
    top_k: int = 5
    
    def _get_relevant_documents(self, query, *, run_manager) -> List[Document]:
        from collections import defaultdict
        scores: dict[str, tuple[Document, float]] = {}
        
        for retriever, weight in zip(self.retrievers, self.weights):
            docs = retriever.get_relevant_documents(query)
            for rank, doc in enumerate(docs):
                key = doc.page_content[:100]
                rrf_score = weight / (60 + rank + 1)  # RRF formula
                if key in scores:
                    scores[key] = (doc, scores[key][1] + rrf_score)
                else:
                    scores[key] = (doc, rrf_score)
        
        ranked = sorted(scores.values(), key=lambda x: x[1], reverse=True)
        return [doc for doc, _ in ranked[:self.top_k]]

print('Custom retrievers extend BaseRetriever:')
print('  APIRetriever: fetch from REST endpoints')
print('  WeightedEnsembleRetriever: RRF fusion of multiple retrievers')
print()
print('Interface contract: implement _get_relevant_documents(query) → List[Document]')
print('Then use in any LCEL chain: retriever | format_docs | prompt | llm | parser')

## 3. Output Parsers — Pydantic, JSON, Structured

In [ ]:
from langchain.output_parsers import PydanticOutputParser, CommaSeparatedListOutputParser
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from typing import Optional

# Pydantic output parser
class JobPosting(BaseModel):
    title: str = Field(description='Job title')
    company: str = Field(description='Company name')
    skills: list[str] = Field(description='Required technical skills')
    salary_min: Optional[int] = Field(description='Minimum salary in USD', default=None)
    salary_max: Optional[int] = Field(description='Maximum salary in USD', default=None)
    remote: bool = Field(description='Whether the job is remote')

parser = PydanticOutputParser(pydantic_object=JobPosting)

extraction_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You extract structured information from job postings.'),
    ('user', 'Extract info from this job posting:\n\n{posting}\n\n{format_instructions}'),
]).partial(format_instructions=parser.get_format_instructions())

job_chain = extraction_prompt | llm | parser

# With automatic retry on parse failure
from langchain.output_parsers import RetryWithErrorOutputParser

retry_parser = RetryWithErrorOutputParser.from_llm(parser=parser, llm=llm, max_retries=3)

# Sample output to show expected structure
sample_output = JobPosting(
    title='Senior AI Engineer',
    company='TechCorp',
    skills=['Python', 'LangChain', 'RAG', 'vector databases'],
    salary_min=150000, salary_max=200000,
    remote=True,
)

print('Parsed job posting structure:')
print(sample_output.model_dump_json(indent=2))
print()
print('Format instructions injected into prompt:')
print(parser.get_format_instructions()[:300] + '...')

## 4. Document Transformers and Post-Processing

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
from langchain_core.documents import Document
from langchain.retrievers.document_compressors import LLMChainExtractor, EmbeddingsFilter
from langchain.retrievers import ContextualCompressionRetriever
from langchain_openai import OpenAIEmbeddings

# 1. Markdown-aware splitter
md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "H1"), ("##", "H2"), ("###", "H3")],
    strip_headers=False,
)

# 2. Contextual compression: extract only relevant sentences
def build_compression_retriever(base_retriever, use_llm: bool = False):
    if use_llm:
        # LLM reads each retrieved chunk and extracts only relevant sentences
        compressor = LLMChainExtractor.from_llm(llm)
    else:
        # Embedding-based filter: keep chunks with similarity > threshold
        embeddings = OpenAIEmbeddings(api_key=os.getenv('OPENAI_API_KEY', 'sk-...'))
        compressor = EmbeddingsFilter(embeddings=embeddings, similarity_threshold=0.76)
    
    return ContextualCompressionRetriever(
        base_compressor=compressor,
        base_retriever=base_retriever,
    )

# 3. Document post-processor
def format_docs_with_metadata(docs: list[Document]) -> str:
    """Format retrieved docs for LLM context with source citation."""
    parts = []
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get('source', 'unknown')
        parts.append(f'[{i}] {doc.page_content}\n(Source: {source})')
    return '\n\n'.join(parts)

# Splitting strategies comparison
splitters = [
    ('RecursiveCharacter', 'General text', 'splits on \\n\\n, \\n, sentences, words'),
    ('MarkdownHeader', 'Documentation/wikis', 'preserves document structure'),
    ('Token', 'LLM context-aware', 'splits by actual token count'),
    ('Semantic', 'Highest quality', 'embeds sentences, splits at semantic boundaries'),
    ('HTML', 'Web pages', 'splits by HTML tags'),
]
print('Text splitter comparison:')
for name, best_for, method in splitters:
    print(f'  {name:<22}: best for {best_for:<22} — {method}')

## 5. Building a Complex Multi-Step Chain

In [ ]:
from langchain_core.runnables import RunnableBranch, RunnablePassthrough
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

# Routing chain: route to different sub-chains based on query type
ROUTER_PROMPT = ChatPromptTemplate.from_template(
    """Classify this user question into one category:
- FACTUAL: asking for a specific fact or definition
- COMPARISON: comparing two or more things
- PROCEDURAL: asking how to do something
- OTHER: anything else

Question: {query}
Category (one word):"""
)

router_chain = ROUTER_PROMPT | llm | StrOutputParser()

factual_prompt = ChatPromptTemplate.from_template('Answer this factual question concisely: {query}')
comparison_prompt = ChatPromptTemplate.from_template('Compare these items with a structured table: {query}')
procedural_prompt = ChatPromptTemplate.from_template('Give step-by-step instructions for: {query}')

# Branch to different sub-chains based on route
routed_chain = RunnableBranch(
    (lambda x: 'FACTUAL' in x['route'], factual_prompt | llm | StrOutputParser()),
    (lambda x: 'COMPARISON' in x['route'], comparison_prompt | llm | StrOutputParser()),
    (lambda x: 'PROCEDURAL' in x['route'], procedural_prompt | llm | StrOutputParser()),
    ChatPromptTemplate.from_template('Answer: {query}') | llm | StrOutputParser(),  # default
)

full_chain = (
    RunnablePassthrough.assign(route=router_chain)
    | routed_chain
)

# Test routing
test_queries = [
    ('What is BERT?', 'FACTUAL'),
    ('Compare BERT vs GPT', 'COMPARISON'),
    ('How do I fine-tune LLaMA?', 'PROCEDURAL'),
]
print('Routing chain demo:')
for query, expected_route in test_queries:
    print(f'  Query: "{query}"')
    print(f'  Expected route: {expected_route}')
    print(f'  → Routes to specialized prompt + handles differently')
    print()

## Exercises

1. **Streaming chain:** Build an LCEL chain that streams output token by token to a WebSocket using FastAPI's WebSocket support. Test with a long-form generation task.

2. **Dynamic routing:** Extend the routing chain to use a vector store of example queries (one per route). At runtime, find the closest example via similarity search and route accordingly — avoiding LLM classification entirely.

3. **Chain inspection:** Use LangChain's `.get_graph()` method to visualize the DAG of your full RAG chain. Export it as a Mermaid diagram and embed in a Jupyter notebook.